In [ ]:
!uv pip install unsloth trl

In [ ]:
import json
from datasets import load_dataset

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
def format_for_training(examples: dict[str, list[str]], source_lang_code: str = "en",
                        target_lang_code: str = "mos", source_field: str = "source",
                        target_field: str = "target"):
    texts = []
    src = examples[source_field]
    tgt = examples[target_field]
    for source, target in zip(src, tgt, strict=True):
        json_str = json.dumps(
            [
                {
                    "type": "text",
                    "source_lang_code": source_lang_code,
                    "target_lang_code": target_lang_code,
                    "text": source,
                }
            ],
            ensure_ascii=False,
        )
        prompt = f"user\n{json_str}\nmodel\n{target}"
        texts.append(prompt)
    return {"text": texts}


def format_for_prediction(examples: dict[str, list[str]], source_lang_code: str = "en",
                          target_lang_code: str = "mos", source_field: str = "source"):
    texts = []
    src = examples[source_field]
    for source in src:  # fix: was `for source, target in src`
        json_str = json.dumps(
            [
                {
                    "type": "text",
                    "source_lang_code": source_lang_code,
                    "target_lang_code": target_lang_code,
                    "text": source,
                }
            ],
            ensure_ascii=False,
        )
        prompt = f"user\n{json_str}\nmodel\n"
        texts.append(prompt)
    return {"text": texts}
def format_single_for_prediction(
    source_text: str,
    source_lang_code: str = "en",
    target_lang_code: str = "mos"
):
    json_str = json.dumps([
        {
            "type": "text",
            "source_lang_code": source_lang_code,
            "target_lang_code": target_lang_code,
            "text": source_text
        }
    ])

    prompt = f"user\n{json_str}\nmodel\n"

    return prompt

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/translategemma-4b-it",
    max_seq_length = 256, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
from transformers import TextStreamer

text_to_translate = "Ring me ra maneg n zemsa ne bʋ-kaoodbã ne wagsda sekirite sosiete, sẽn n yaa ADT Corporation."
prompt = format_single_for_prediction(
    source_text=text_to_translate,
    source_lang_code="mos",
    target_lang_code="en"
)
print(f"Prompt: {prompt}")


FastLanguageModel.for_inference(model)
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **input_ids,
    max_new_tokens=120,
    temperature=0.1,
    top_p=0.8,
    top_k=20,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

# Decode the output to get the translated text
# The streamer already prints the output, but we can also decode it
# translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
# print(f"Translated text: {translated_text}")

In [ ]:
dataset = load_dataset("madoss/fr-mos-final-data")

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_mlp_modules= True,
    finetune_attention_modules = True,
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients


In [ ]:
dataset["train"][0]

{'src': 'Dis : "Adorez-vous, en dehors d’Allah, des créatures qui n’ont aucun pouvoir, ne pouvant ni vous nuire, ni vous être utiles, alors que c’est Allah qui entend tout et sait tout ? »',
 'trg': 'Yeele:" yãmb tũudɑ zẽng sẽn pɑ Wẽnde sẽn pɑ tõe n nɑms-yã, pɑ nɑfgo, lɑ Wẽnde, Yẽ lɑ a Wʋmd n yɑɑ a Mita".',
 'source': 'coran',
 'source_lang': 'fr',
 'target_lang': 'moo'}

In [ ]:
from functools import partial

format_func = partial(format_for_training,  source_lang_code= "fr",
                        target_lang_code = "mos", source_field= "french",
                        target_field= "moore")

dataset = dataset["train"].map(format_func, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
dataset[0]

{'src': 'Dis : "Adorez-vous, en dehors d’Allah, des créatures qui n’ont aucun pouvoir, ne pouvant ni vous nuire, ni vous être utiles, alors que c’est Allah qui entend tout et sait tout ? »',
 'trg': 'Yeele:" yãmb tũudɑ zẽng sẽn pɑ Wẽnde sẽn pɑ tõe n nɑms-yã, pɑ nɑfgo, lɑ Wẽnde, Yẽ lɑ a Wʋmd n yɑɑ a Mita".',
 'source': 'coran',
 'source_lang': 'fr',
 'target_lang': 'moo',
 'text': 'user\n[{"type": "text", "source_lang_code": "fr", "target_lang_code": "trg", "text": "Dis : \\"Adorez-vous, en dehors d’Allah, des créatures qui n’ont aucun pouvoir, ne pouvant ni vous nuire, ni vous être utiles, alors que c’est Allah qui entend tout et sait tout ? »"}]\nmodel\nYeele:" yãmb tũudɑ zẽng sẽn pɑ Wẽnde sẽn pɑ tõe n nɑms-yã, pɑ nɑfgo, lɑ Wẽnde, Yẽ lɑ a Wʋmd n yɑɑ a Mita".'}

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
trainer_stats = trainer.train()

In [ ]:
from transformers import TextStreamer

text = format_single_for_prediction(source_text="Comment allez-vous",
                                    source_lang_code="fr")

_ = model.generate(
    **tokenizer(text=text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 100, # Increase for longer outputs!
    temperature = 0.1, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

user
[{"type": "text", "source_lang_code": "fr", "target_lang_code": "mos", "text": "Comment allez-vous"}]
model

{'input_ids': tensor([[     2,   2364,    107, 236840,  14937,   2084,   1083,    623,   1005,
            827,    623,   6265, 236779,  10694, 236779,   3970,   1083,    623,
           1145,    827,    623,   5654, 236779,  10694, 236779,   3970,   1083,
            623,   8923,    827,    623,   1005,   1083,    623,  22217,  99219,
         236772,  23841, 236775,  15947,    107,   4368,    107]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])}
Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bãmbãmbã? Bã


In [ ]:
model.save_pretrained_merged("translategemma-4b-it_16bit", tokenizer, save_method = "merged_16bit",)

In [ ]:
model.push_to_hub("madoss/translategemma-4b-it-FR-MOS")
tokenizer.push_to_hub("madoss/translategemma-4b-it-FR-MOS")